In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 베어링 결함 진단 — Kurtogram + Bandpass Envelope + MED

베어링 결함 신호는 공진 대역에서 주기적 임펄스로 변조되어 나타난다. Kurtogram 으로 최적 대역을 찾아 밴드패스 필터링한 뒤 힐베르트 포락선 스펙트럼으로 결함 주파수를 드러낸다. 마지막으로 MED (Minimum Entropy Deconvolution) 를 적용해 임펄스 성분을 강조한다.

## 학습 목표
- 베어링 결함 특성 주파수 (BPFO, BPFI) 계산
- Kurtogram 으로 결함 공진 대역 탐지
- 밴드패스 + 힐베르트 포락선 스펙트럼으로 결함 주파수 확인
- MED 적용 전/후 임펄스 강조 효과 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import kurtogram
from utils import fft, filtering, filtering_zerophase, hilbert_envelope, MEDA

---

## 실습 1. 베어링 특성 주파수 계산

CWRU Drive End 베어링 계수:
- BPFI = 5.4152 × f_shaft (내륜)
- BPFO = 3.5848 × f_shaft (외륜)

In [ ]:
rpm = 1772
f_shaft = rpm / 60
DE_BPFI = 5.4152 * f_shaft
DE_BPFO = 3.5848 * f_shaft
print(f'Shaft Frequency : {f_shaft:.2f} Hz')
print(f'DE BPFI (Inner) : {DE_BPFI:.2f} Hz')
print(f'DE BPFO (Outer) : {DE_BPFO:.2f} Hz')

---

## 실습 2. 데이터 로드 — 내륜 결함 신호

`data_fault_DE_IR.csv` — Drive End 내륜(IR) 결함 신호 (fs = 12000 Hz).

In [ ]:
fs = 12000
data = np.array(pd.read_csv('./data/data_fault_DE_IR.csv'))
y = data[:, 1]
N = len(y)
t = np.arange(N) / fs
print(f'Number of samples N = {N}, duration = {N/fs:.3f} s')

# --- 원 신호 시각화 ---
f_y, A_y = fft(y, fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y, color='C0')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('y')
ax1.set_title('Raw signal (IR fault)')
ax2.plot(f_y, A_y, color='C0')
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('|Y|')
ax2.set_title('Spectrum')
fig.tight_layout()
plt.show()

원 신호 스펙트럼에서는 결함 주파수(약 160 Hz)를 직접 확인하기 어렵다. 결함 임펄스가 공진 대역으로 변조되어 있기 때문이다.

---

## 실습 3. Kurtogram — 결함 공진 대역 탐지

Spectral Kurtosis를 여러 해상도(level)에서 계산하여 충격성(임펄스)이 가장 강한 주파수 대역을 찾는다.

(`nlevel=7`은 kurtogram의 이진 트리 분해 깊이를 의미한다 — 주파수 축이 최대 $2^7 = 128$개의 대역으로 분해된다. 데이터 길이 $N$에 대해 $\log_2 N - 7 > 0$을 만족해야 한다.)

In [ ]:
nlevel = 7
Kwav, Level_w, freq_w, c, max_Kurt, bandwidth, level_max = kurtogram.fast_kurtogram(y.copy(), fs, nlevel=nlevel)

minw = np.where(Level_w == level_max)[0][0]
kurtw = np.where(Kwav[minw, :] == max_Kurt)[0][0]
bandw = freq_w[kurtw]
f_center = bandw + bandwidth / 2

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(Kwav, interpolation='none', aspect='auto')
xticks = np.linspace(0, fs/2, nlevel + 1).astype(int)
ax.set_xticks(np.linspace(0, Kwav.shape[1] - 1, nlevel + 1))
ax.set_xticklabels(xticks)
ax.set_yticks(np.arange(0, Kwav.shape[0]))
ax.set_yticklabels(np.round(Level_w, 1))
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Level')
ax.set_title(f'K max = {max_Kurt:.3f} at level {level_max:.1f}\n'
             f'Center = {f_center:.1f} Hz, BW = {bandwidth:.1f} Hz')
fig.colorbar(im, ax=ax, label='Spectral Kurtosis')
fig.tight_layout()
plt.show()

---

## 실습 4. 밴드패스 필터링 — 탐지된 공진 대역

Kurtogram이 찾은 [bandw, bandw+bandwidth] 구간을 Butterworth 대역통과로 추출한다.

In [ ]:
# Kurtogram 이 선택한 대역을 bandpass 로 사용
# — Nyquist 경계를 넘으면 fs/2 바로 아래로 clamp
b1 = max(bandw, 1.0)
b2 = bandw + bandwidth
if b2 > fs / 2:
    b2 = fs / 2 - 1
f_low, f_high = b1, b2
print(f'Bandpass: [{f_low:.1f}, {f_high:.1f}] Hz')

y_bp = filtering_zerophase(y, fs, 'band', f_low=f_low, f_high=f_high, order=4)
f_bp, A_bp = fft(y_bp, fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y_bp, color='C1')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Amplitude')
ax1.set_title(f'Bandpass [{f_low:.0f}, {f_high:.0f}] Hz — time domain')

ax2.plot(f_bp, A_bp, color='C1')
ax2.set_xlim(0, fs/2); ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('Amplitude')
ax2.set_title('Bandpass FFT')
plt.tight_layout(); plt.show()

---

## 실습 5. 힐베르트 포락선 스펙트럼

대역통과 후 포락선(= 해석신호의 크기)을 구하고 FFT를 취하면 변조 주파수(= 결함 주파수)가 드러난다.

In [ ]:
env_bp = hilbert_envelope(y_bp)
f_env, A_env = fft(env_bp - np.mean(env_bp), fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y_bp, color='C1', label='bandpass')
ax1.plot(t, env_bp, color='C3', label='envelope')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('amp')
ax1.legend()

ax2.plot(f_env, A_env, color='C3')
ax2.set_xlim(0, 500)
for n in range(1, 4):
    ax2.axvline(n * DE_BPFI, color='C2', linestyle='--', alpha=0.7,
                label='BPFI harmonics' if n == 1 else None)
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('|Env|')
ax2.set_title('Envelope Spectrum (dashed: BPFI Harmonics)')
ax2.legend()
fig.tight_layout()
plt.show()

포락선 스펙트럼에서 BPFI(약 160 Hz)와 그 배음이 드러나면 내륜 결함으로 진단할 수 있다 — 공진 대역을 정확히 고른 경우에 한함 (대역이 어긋나면 shaft rate 성분이 지배적으로 보인다).

---

## 실습 6. MED (Minimum Entropy Deconvolution)

MED는 신호를 가장 "임펄스적"으로 만드는 필터를 데이터로부터 학습한다.
밴드 선택 없이 원 신호에서 바로 임펄스를 강조하는 대안 기법이다.

In [ ]:
FilterSize = 50
_, y_MED = MEDA(y, FilterSize=FilterSize, remain=True)
y_MED[:FilterSize] = 0

f_MED, A_MED = fft(y_MED, fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y_MED, color='C4')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('y_MED')
ax1.set_title(f'MED filtered (FilterSize={FilterSize})')
ax2.plot(f_MED, A_MED, color='C4')
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('|Y|')
ax2.set_title('Spectrum of MED output')
fig.tight_layout()
plt.show()

**관찰 (MED)**: MED 출력 `y_MED` 는 원 신호의 임펄스성 쇼크를 강조하도록 최적화된 FIR 역필터를 곱한 결과다. 시간 파형에서 각 임펄스의 **고립도**가 bandpass 결과보다 더 커지지만, shaft rate 오염이나 공진 대역 외 성분은 그대로 남을 수 있어 이후 포락선 분석에서 shaft rate 쪽 피크가 강하게 보일 수 있다.

**주**: FIR 필터 길이 (`FilterSize`) 는 신호 특성에 따라 튜닝이 필요하다. 이 데이터에서는 `FilterSize=50` 이 임펄스 주기 (~75 샘플) 와 맞아떨어져 잘 동작한다. 너무 길면 (~100 이상) MED 가 신호 구조를 지나치게 평활화해 BPFI 변조가 사라진다.

In [ ]:
env_MED = hilbert_envelope(y_MED)
f_menv, A_menv = fft(env_MED - np.mean(env_MED), fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y_MED, color='C4', label='MED')
ax1.plot(t, env_MED, color='C3', label='envelope')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('amp')
ax1.legend()

ax2.plot(f_menv, A_menv, color='C3')
ax2.set_xlim(0, 500)
for n in range(1, 4):
    ax2.axvline(n * DE_BPFI, color='C2', linestyle='--', alpha=0.7,
                label='BPFI harmonics' if n == 1 else None)
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('|Env|')
ax2.set_title('Envelope spectrum of MED output')
ax2.legend()
fig.tight_layout()
plt.show()

In [ ]:
# 5단계 비교 — raw / bandpass / BP envelope / MED / MED envelope (5-stage comparison)
fig, axes = plt.subplots(5, 1, figsize=(13, 10), sharex=True)
panels = [
    ('Raw',          y,         'C0'),
    ('SK+BP',        y_bp,     'C1'),
    ('SK+BP env',    env_bp,   'C3'),
    ('MED',          y_MED,    'C4'),
    ('MED envelope', env_MED,  'C2'),
]
for ax, (label, sig, color) in zip(axes, panels):
    t_plot = np.arange(len(sig)) / fs
    ax.plot(t_plot, sig, color=color, lw=0.6)
    ax.set_ylabel('Amplitude'); ax.set_title(label)
axes[-1].set_xlabel('Time (s)')
axes[0].set_xlim(0, 0.2)
fig.suptitle('5-Stage Time-Domain Comparison')
plt.tight_layout(); plt.show()

### 두 경로의 포락선 스펙트럼 비교

Bandpass + Envelope 경로와 MED + Envelope 경로에서 얻은 포락선 스펙트럼을 나란히 비교해 BPFI 배음 정렬을 확인한다.

In [ ]:
# 두 경로의 포락선 스펙트럼 비교
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
f_cmp_bp, A_cmp_bp = fft(env_bp - np.mean(env_bp), fs)
f_cmp_med, A_cmp_med = fft(env_MED - np.mean(env_MED), fs)

ax[0].plot(f_cmp_bp, A_cmp_bp, 'C1')
ax[0].set_title('Spectral Kurtosis + Band-pass + Envelope')
ax[0].set_xlim(0, 500)

ax[1].plot(f_cmp_med, A_cmp_med, 'C4')
ax[1].set_title('MED + Envelope')
ax[1].set_xlim(0, 500)

for a in ax:
    for k in range(1, 6):
        a.axvline(k * DE_BPFI, color='C2', ls='--', alpha=0.5)
    a.set_xlabel('Frequency (Hz)')
    a.set_ylabel('Amplitude')

fig.tight_layout()
plt.show()

# 정량 비교: BPFI 피크 대 배경 ratio
def peak_ratio(f, A, f0, df=2.0):
    """피크(=f0±df) / 넓은 주변 floor(=f0±5df 를 제외한 f0-50 ~ f0+50)."""
    m_peak = (f > f0 - df) & (f < f0 + df)
    peak = A[m_peak].max() if m_peak.any() else 0.0
    m_floor = ((f > f0 - 50) & (f < f0 + 50)
               & ~((f > f0 - 5*df) & (f < f0 + 5*df)))
    floor = np.median(A[m_floor]) if m_floor.any() else 1e-12
    return peak / max(floor, 1e-12)

print(f'BPFI peak/floor — BP envelope  : {peak_ratio(f_cmp_bp,  A_cmp_bp,  DE_BPFI):.2f}')
print(f'BPFI peak/floor — MED envelope : {peak_ratio(f_cmp_med, A_cmp_med, DE_BPFI):.2f}')

## 정리

- **Kurtogram**: 결함 임펄스가 실린 최적 공진 대역을 자동 탐지
- **Bandpass + Hilbert envelope**: 탐지된 대역에서 변조 주파수(결함 주파수)를 드러냄
- **MED**: 데이터 기반 필터로 임펄스 성분을 직접 강조
- 두 경로 모두 포락선 스펙트럼에서 BPFI 및 그 배음이 나타남을 확인

### 생각해보기
1. Kurtogram의 `nlevel`을 바꾸면 찾은 중심주파수/대역폭이 어떻게 달라지는가?
2. MED의 `FilterSize`를 10, 100, 500으로 바꾸면 결과 임펄스 선명도는 어떻게 변하는가?
3. 외륜(OR) 결함 신호(`data_fault_FE_OR.csv`)에 동일 파이프라인을 적용하면 어떤 주파수가 드러나야 하는가?
